In [ ]:
#Visualization 1
#This script performs correlation analysis and Mantel tests between yield components and environmental / socioeconomic features.
#The outputs are used to generate the network-correlation heatmap shown in Figure 2.
#The workflow includes:
#    - Load standardized yield and feature datasets
#    - Perform Mantel tests between yield components and features
#    - Compute Pearson correlation matrix of features
#    - Construct a combined network-correlation heatmap
#    - Export datasets and final visualization

### Load required libraries.
library(linkET)
suppressPackageStartupMessages(library(tidyverse))

### Load datasets.
yield <- read_csv("../results/book/2/yield_dataset.csv", show_col_types = FALSE)
feature <- read_csv("../results/book/2/feat_dataset.csv", show_col_types = FALSE)

### Mantel test between yield components and feature set.
mantel <- mantel_test(yield, feature,
    spec_dist = "euclidean",
    env_dist = "euclidean",
    spec_select = list("YO" = 1, "TY" = 2, "DY" = 3),
    permutations = 999
) %>%
    mutate(
        rd = cut(abs(r),
            breaks = c(-Inf, 0.15, Inf),
            labels = c("<= 0.15", "> 0.15")
        ),
        pd = cut(p,
            breaks = c(-Inf, 0.005, 0.01, 0.05, Inf),
            labels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05"),
        )
    )
write_csv(mantel, "../results/book/2/mantel_result.csv")


In [ ]:
### Load additional visualization libraries.
library(linkET)
library(ggnewscale)
library(RColorBrewer)
library(viridis)
suppressPackageStartupMessages(library(tidyverse))

### Reload datasets for visualization.
feature <- read_csv("../results/book/2/feat_dataset.csv", show_col_types = FALSE)
mantel <- read_csv("../results/book/2/mantel_result.csv", show_col_types = FALSE) %>%
    mutate(pd = factor(pd, levels = c("<= 0.005", "0.005 - 0.01", "0.01 - 0.05", "> 0.05")))

### Compute Pearson correlation matrix for feature variables.
cor_mat <- correlate(feature)
cor_mat <- unclass(cor_mat)
cor_mat <- as.data.frame(cor_mat)
cor_mat <- tibble::rownames_to_column(cor_mat, var = "Var")
write_csv(cor_mat, "../results/book/2/correl_matrix.csv")

### Construct correlation heatmap with Mantel network overlay.
p2 <- qcorrplot(correlate(feature), type = "upper", diag = FALSE, grid_col = NA) +
    geom_point(shape = 21, size = 4, fill = NA, stroke = 0.5, color = "black") +
    geom_point(aes(size = abs(r), fill = r),
        shape = 21,
        stroke = 0.4,
        color = "black"
    ) +
    scale_size(range = c(1, 3), guide = "none") +
    new_scale("size") +
    geom_couple(
        data = mantel,
        aes(color = pd, size = rd),
        label.size = 2.46,
        label.family = "Arail",
        label.fontface = 2,
        nudge_x = 0.5,
        curvature = nice_curvature(by = "from")
    ) +
    scale_fill_gradientn(
        limits = c(-0.8, 0.8),
        breaks = seq(-0.8, 0.8, 0.4),
        colors = rev(brewer.pal(11, "RdBu"))
    ) +
    scale_size_manual(values = c(0.3, 1)) +
    scale_color_manual(values = viridis(8, alpha = 0.88)) +
    guides(
        fill = guide_colorbar(
            title = "Pearson's r",
            title.vjust = 3,
            keyheight = unit(1.8, "cm"),
            keywidth = unit(0.3, "cm"),
            order = 1
        ),
        size = guide_legend(
            title = "Mantel's r",
            order = 2,
            keyheight = unit(0.3, "cm")
        ),
        colour = guide_legend(
            title = "Mantel's p",
            order = 3,
            keyheight = unit(0.3, "cm")
        )
    ) +
    theme(
        legend.box.spacing = unit(3, "pt"),
        axis.text = element_text(size = 6),
        legend.title = element_text(size = 6),
        legend.text = element_text(size = 5)
    )
ggsave(p2, file = "../results/book/2/net_heat_plot.svg", width = 5, height = 4)
